<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Roteiro de Atividade Prática</h2>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Projeto Análise Exploratória de Dados  
**Tema da Semana:** Projeto AED – Coleta e Preparação dos Dados  
**Semana 7 — Aula 4:** Validação da base preparada para análise

**Nome(s):** [preencher]  
**Turma:** [preencher]

**Objetivo:** Analisar a adequação da base preparada às perguntas e aos objetivos definidos para a investigação.


## Missão da atividade

Vocês atuam como assistentes de dados da equipe comercial de uma pequena rede brasileira de papelarias. A base é simulada e cada registro representa a operação de uma loja em um dia.

**Pergunta central:** Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?

A S6 indicou a necessidade de aprofundar a investigação por **loja e faixa de movimento**. A redução da espera apareceu como um exemplo de prioridade fundamentada, não como conclusão obrigatória. Nesta semana, os dados foram selecionados, verificados e preparados.

**Missão:** Validar para quais usos a base preparada permite prosseguir, sob quais condições e quais pendências ainda precisam ser aprofundadas antes de ampliar as conclusões.

Trabalhem em duplas. Em cada etapa, um integrante executa as células e o outro confere as saídas e a justificativa. Alternem os papéis antes de registrar a decisão.

**Tempo total: 43 minutos.** As etapas duram 4, 8, 9, 10, 7 e 5 minutos.

A prioridade discutida na S6 não precisa ser confirmada. As novas evidências podem confirmá-la, qualificá-la ou levar à sua revisão. Associação não demonstra causalidade, e os resultados se restringem ao conjunto simulado.


## 1. Conheçam os critérios de validação — 4 minutos

Arquivos usados nesta atividade:

- `DADOS3EMC3B2S7A4_aluno.ipynb`
- `rede_papelarias_B2_integrado.csv`

Deixem o notebook e o CSV na mesma pasta. No Visual Studio Code, selecionem um kernel Python disponível. No Google Colab, façam o upload do CSV para a sessão. Executem as células na ordem e **não alterem o código**.

O notebook recompõe uma **base preparada de referência** usando apenas procedimentos já conhecidos: retira a cópia integral confirmada e recupera `cupons_por_100_entradas` e `margem_bruta_pct`. Nenhuma ausência é preenchida, nenhuma despesa é convertida e nenhum possível valor atípico é removido.

A validação usa três classificações:

- **Pode prosseguir:** os dados estão organizados e disponíveis para o uso definido.
- **Pode prosseguir com restrição:** o uso é possível, mas há uma condição ou limite que precisa acompanhar a interpretação.
- **Ainda depende de pendência:** falta resolver um problema necessário para aquele uso.

Durante as etapas 2 a 5, sustentem cada decisão com **evidência da saída + finalidade + limite**.


In [ ]:
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(valor):
        print(valor)

nome_preferencial = "rede_papelarias_B2_integrado.csv"
arquivo_csv = Path(nome_preferencial)

if not arquivo_csv.is_file():
    candidatos = sorted(Path.cwd().glob("*rede_papelarias_B2_integrado*.csv"))
    if len(candidatos) != 1:
        raise FileNotFoundError(
            "Deixe o CSV integrado desta atividade na mesma pasta do notebook "
            "ou faça o upload para a sessão do Colab."
        )
    arquivo_csv = candidatos[0]

dados_originais = pd.read_csv(arquivo_csv)

# Base preparada de referência: recompõe somente procedimentos já conhecidos.
dados_preparados = dados_originais.drop_duplicates().copy()

if (dados_preparados["entradas"] <= 0).any() or (dados_preparados["receita_liquida"] <= 0).any():
    raise ValueError("Há denominadores não positivos; interrompa e avise o professor.")

dados_preparados["cupons_por_100_entradas"] = (
    100 * dados_preparados["cupons_emitidos"] / dados_preparados["entradas"]
)
dados_preparados["margem_bruta_pct"] = (
    100 * dados_preparados["lucro_bruto"] / dados_preparados["receita_liquida"]
)

faixa_inicio, faixa_fim = 150, 199
dados_faixa = dados_preparados[
    dados_preparados["entradas"].between(faixa_inicio, faixa_fim)
].copy()

print("Arquivo carregado:", arquivo_csv.name)
print("Registros originais:", len(dados_originais))
print("Registros na base preparada:", len(dados_preparados))
print("Colunas originais:", dados_originais.shape[1])
print("Colunas na base preparada:", dados_preparados.shape[1])
print("Faixa de movimento retomada do projeto: 150–199 entradas")


## 2. Validem as comparações por loja e faixa de movimento — 8 minutos

A S6 mostrou que resultados globais não descrevem automaticamente o que ocorre em cada loja. Por isso, o aprofundamento precisa considerar **loja** e uma **faixa comum de movimento**.

A faixa de **150–199 entradas** já foi utilizada no projeto porque apresentou a melhor cobertura mínima por loja entre as faixas comparadas. Nesta aula, não escolham outra faixa: verifiquem se o estado preparado da base continua sustentando esse uso.


In [ ]:
ordem_lojas = sorted(dados_preparados["loja"].dropna().unique())

cobertura = pd.DataFrame({
    "registros_totais": dados_preparados.groupby("loja").size().reindex(ordem_lojas),
    "registros_150_199": dados_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0),
    "ausentes_cupons_por_100_na_faixa": dados_faixa.groupby("loja")["cupons_por_100_entradas"].apply(lambda s: int(s.isna().sum())).reindex(ordem_lojas, fill_value=0),
    "ausentes_margem_bruta_na_faixa": dados_faixa.groupby("loja")["margem_bruta_pct"].apply(lambda s: int(s.isna().sum())).reindex(ordem_lojas, fill_value=0),
})

print("Cobertura por loja:")
display(cobertura)
print("Total de registros na faixa 150–199:", len(dados_faixa))
print("Menor número de registros entre as lojas nessa faixa:", int(cobertura["registros_150_199"].min()))
print("Lojas representadas na faixa:", int((cobertura["registros_150_199"] > 0).sum()), "de", len(ordem_lojas))


### Registro — Q1

**A base preparada permite aprofundar comparações por loja na faixa de 150–199 entradas?**

- **Classificação:** [Pode prosseguir / Pode prosseguir com restrição / Ainda depende de pendência]
- **Evidência 1:** [preencher]
- **Evidência 2:** [preencher]
- **Condição ou limite:** [preencher]

Não concluam que uma diferença observada dentro dessa faixa representa automaticamente todas as condições de operação das lojas.


## 3. Validem a investigação sobre espera e aproveitamento — 9 minutos

A S6 encontrou uma associação relevante entre espera e aproveitamento do movimento. Para aprofundá-la por loja e faixa de movimento, precisamos saber se há pares observados de `espera_media_min` e `cupons_por_100_entradas` em quantidade suficiente para continuar a investigação.

Nesta etapa, **não calculem correlação nem façam gráfico**. O objetivo é avaliar a adequação dos dados à análise futura.


In [ ]:
par_espera = ["espera_media_min", "cupons_por_100_entradas"]
pares_base = dados_preparados.dropna(subset=par_espera)
pares_faixa = dados_faixa.dropna(subset=par_espera)

cobertura_espera_faixa = pd.DataFrame({
    "registros_na_faixa": dados_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0),
    "pares_espera_aproveitamento": pares_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0),
})
cobertura_espera_faixa["espera_ausente_na_faixa"] = (
    cobertura_espera_faixa["registros_na_faixa"] - cobertura_espera_faixa["pares_espera_aproveitamento"]
)

resumo_espera = pd.DataFrame({
    "situação": [
        "Base preparada",
        "Pares observados na base",
        "Faixa 150–199",
        "Pares observados na faixa",
    ],
    "registros": [
        len(dados_preparados),
        len(pares_base),
        len(dados_faixa),
        len(pares_faixa),
    ],
})

print("Cobertura da espera:")
display(resumo_espera)
print("Por loja, na faixa 150–199:")
display(cobertura_espera_faixa)
print("Esperas ausentes na base preparada:", int(dados_preparados["espera_media_min"].isna().sum()))
print("Esperas ausentes na faixa 150–199:", int(dados_faixa["espera_media_min"].isna().sum()))
print("Menor número de pares observados por loja na faixa:", int(cobertura_espera_faixa["pares_espera_aproveitamento"].min()))


### Registro — Q2

**Os dados permitem investigar espera × aproveitamento por loja e faixa de movimento?**

- **Classificação:** [Pode prosseguir / Pode prosseguir com restrição / Ainda depende de pendência]
- **Evidência:** [preencher]
- **Como as ausências limitam a abrangência:** [preencher]
- **O que deve ser preservado:** [preencher]

Lembrem-se: espera ausente significa **informação desconhecida**, não espera igual a zero.


## 4. Validem o aprofundamento do resultado comercial — 10 minutos

A base ampliada acrescentou informações de despesas. Agora precisamos distinguir três usos diferentes:

1. comparar **margem bruta**;
2. investigar um **resultado após despesas operacionais**;
3. decidir como tratar um **valor de receita muito distante dos demais**.

Executem a conferência abaixo. Não convertam despesas, não preencham ausências e não excluam o registro destacado.


In [ ]:
# Margem bruta: disponibilidade para uso analítico já conhecido.
resumo_margem = pd.DataFrame({
    "variavel": ["margem_bruta_pct"],
    "valores_disponiveis": [int(dados_preparados["margem_bruta_pct"].notna().sum())],
    "ausentes": [int(dados_preparados["margem_bruta_pct"].isna().sum())],
})
print("Margem bruta percentual:")
display(resumo_margem)

# Despesas: somente diagnóstico de disponibilidade e representação.
linhas_despesas = []
for coluna in ["despesas_com_pessoal", "outras_despesas_operacionais"]:
    textos = dados_preparados[coluna].dropna().astype(str)
    linhas_despesas.append({
        "coluna": coluna,
        "tipo_lido": str(dados_preparados[coluna].dtype),
        "ausentes": int(dados_preparados[coluna].isna().sum()),
        "valores_com_R$": int(textos.str.contains("R$", regex=False).sum()),
        "valores_com_virgula": int(textos.str.contains(",", regex=False).sum()),
    })
print("Pendências das despesas:")
display(pd.DataFrame(linhas_despesas))
print("Contagens de formato não devem ser somadas às ausências.")

print("Registro de receita destacado no diagnóstico:")
display(
    dados_preparados.loc[
        dados_preparados["id_registro"].eq("R03047"),
        ["id_registro", "data", "loja", "receita_liquida", "lucro_bruto", "margem_bruta_pct"],
    ].round(2)
)


### Registro — Q3

Para cada uso abaixo, registrem **se pode prosseguir agora** e justifiquem com a saída.

**a) Comparar margem bruta entre observações ou grupos**  
Classificação e justificativa: [preencher]

**b) Calcular um resultado após as despesas operacionais acrescentadas à base**  
Classificação e justificativa: [preencher]

**c) Excluir o registro R03047 apenas porque sua receita é muito diferente das demais**  
Classificação e justificativa: [preencher]

**Encaminhamento das pendências:**

- ausências e possível valor atípico → [preencher]
- problemas de representação/formato → [preencher]


## 5. Consolidem o diagnóstico de adequação — 7 minutos

Executem a célula de síntese das evidências e, em seguida, completem o Registro de validação.

A decisão deve variar conforme o **uso pretendido**. Evitem conclusões gerais como “a base está limpa” ou “a base está pronta para qualquer análise”.


In [ ]:
evidencias_validacao = pd.DataFrame([
    {
        "uso": "Comparações por loja e faixa 150–199",
        "evidencia_observada": f"{len(dados_faixa)} loja-dias; 8 lojas; mínimo de {int(cobertura['registros_150_199'].min())} registros por loja",
    },
    {
        "uso": "Espera × aproveitamento na faixa 150–199",
        "evidencia_observada": f"{len(pares_faixa)} pares observados; {int(dados_faixa['espera_media_min'].isna().sum())} esperas ausentes na faixa",
    },
    {
        "uso": "Margem bruta",
        "evidencia_observada": f"{int(dados_preparados['margem_bruta_pct'].notna().sum())} valores disponíveis de margem_bruta_pct",
    },
    {
        "uso": "Resultado após despesas operacionais",
        "evidencia_observada": "despesas com ausências e formatos mistos; cálculo ampliado ainda não preparado",
    },
])

display(evidencias_validacao)


### Registro de validação — Q4

Completem a matriz abaixo.

| Uso na investigação | Evidência de adequação | Decisão | Condição ou limite | Próxima etapa |
|---|---|---|---|---|
| Comparações por loja e faixa 150–199 | [preencher] | [preencher] | [preencher] | [preencher] |
| Espera × aproveitamento | [preencher] | [preencher] | [preencher] | [preencher] |
| Margem bruta | [preencher] | [preencher] | [preencher] | [preencher] |
| Resultado após despesas operacionais | [preencher] | [preencher] | [preencher] | [preencher] |

**Q4. Diagnóstico final:** Em uma frase, diferenciem o que já pode prosseguir do que ainda depende de pendência.  
**Resposta:** [preencher]

Fatores como organização da equipe, perfil do público, mix de produtos, eventos locais e características do atendimento podem continuar relevantes para explicar diferenças, mas não são “valores ausentes” nas colunas atuais. São informações ainda não registradas ou não demonstradas como disponíveis.


## 6. Revisem e comuniquem a validação — 5 minutos

**Revisão entre duplas — 2 min:** troquem o diagnóstico e questionem uma decisão de adequação.  
**Ajustes — 2 min:** melhorem a justificativa ou sustentem sua manutenção com evidência.  
**Síntese — 1 min:** respondam à equipe comercial em que condições a investigação pode continuar.

### Registro da revisão — Q5

**Observação recebida:** [preencher]

**Ajuste realizado ou justificativa para manter a decisão:** [preencher]

### Síntese da aprendizagem — Q6

A equipe comercial pergunta:

> **“Com a base preparada, quais partes do aprofundamento definido no projeto podem prosseguir, sob quais condições e o que ainda precisa ser resolvido?”**

**Resposta:** [preencher]

Não transformem a validação em recomendação final de intervenção. A prioridade discutida anteriormente pode ser confirmada, qualificada ou revista quando novas evidências forem produzidas.


## Critérios para avaliação da atividade

- Analisa a adequação da base em relação às perguntas e aos usos definidos para o aprofundamento da investigação.
- Sustenta decisões de adequação com evidências observáveis de cobertura, disponibilidade e preparação dos dados.
- Distingue usos que podem prosseguir, usos condicionados e usos que ainda dependem de pendências.
- Encaminha ausências e possíveis valores atípicos para aprofundamento na S8 e problemas de representação ou formato para a S9, sem antecipar seus tratamentos.
- Revê a própria decisão com outra dupla e preserva os limites da recomendação, sem afirmar causalidade ou uma prioridade comercial definitiva.

**Registro da atividade:** diagnóstico de adequação, revisão entre duplas e síntese final preenchidos no notebook e discutidos durante esta aula, sem exportação de dados para a próxima aula.

**Fonte dos dados:** base integrada do projeto SEDUC C3 B2, simulada para fins didáticos. O arquivo `rede_papelarias_B2_integrado.csv` é recomposto no notebook como base preparada de referência para validação.


# Expectativa de respostas

Uso exclusivo do professor. O núcleo anterior é integralmente igual ao do aluno.

## Orientações ao professor

A finalidade é levar os estudantes a **analisar a adequação por uso**. A validação não pergunta se a base está genericamente “boa” ou “ruim”; confronta as necessidades concretas do aprofundamento definido no projeto com a evidência disponível.

A S6 indicou a necessidade de aprofundar diferenças por loja e faixa de movimento. A investigação de alternativas para reduzir a espera foi um exemplo de recomendação fundamentada, não uma resposta obrigatória. Não conduza a turma para confirmar essa prioridade: as evidências posteriores podem confirmá-la, qualificá-la ou levar à sua revisão.

**Tempo e gestão:** 43 minutos, distribuídos em 4, 8, 9, 10, 7 e 5 minutos. Trabalhe em duplas, alternando execução e conferência. O notebook recompõe a preparação necessária e não depende de arquivos produzidos pelos estudantes em outra aula.

**Condução geral:** em cada etapa, peça primeiro a finalidade do uso e depois a evidência que sustenta a decisão. Se a dupla responder apenas “tem dados” ou “faltam dados”, retome: quantos registros? em qual variável? para qual comparação? que consequência isso traz?

Não introduza tratamento novo. Ausências e possível valor atípico permanecem para a S8; representação e formatos mistos das despesas permanecem para a S9. Fatores ainda não registrados, como perfil do público e mix de produtos, são lacunas informacionais ou de coleta futura, não defeitos de qualidade da tabela.

## Etapa 1 — Critérios de validação

A base carregada possui **5.001 linhas e 15 colunas**. A preparação de referência remove uma única cópia integral, resultando em **5.000 loja-dias**, e acrescenta `cupons_por_100_entradas` e `margem_bruta_pct`, totalizando **17 colunas**. Nenhuma ausência é preenchida e nenhuma despesa é convertida.

As três classificações devem permanecer ligadas ao uso:

- **Pode prosseguir**: há cobertura e preparação suficientes para aquele uso.
- **Pode prosseguir com restrição**: o uso é possível, mas a limitação deve acompanhar a interpretação.
- **Ainda depende de pendência**: um problema relevante impede aquele uso no estado atual.

## Q1 — Loja e faixa de movimento

**Classificação esperada:** **Pode prosseguir.**

Evidências:

- 5.000 registros após retirar a cópia integral;
- **625 loja-dias em cada uma das oito lojas**;
- a faixa **150–199 entradas** contém **1.741 registros**;
- todas as oito lojas estão representadas;
- o menor número de registros por loja nessa faixa é **115**;
- `cupons_por_100_entradas` e `margem_bruta_pct` não apresentam ausências no recorte.

**Limite:** a faixa é um recorte de movimento semelhante e não representa automaticamente todos os contextos operacionais. A validação permite prosseguir com comparações nesse nível; não demonstra causalidade nem define uma prioridade comercial.

## Q2 — Espera × aproveitamento

**Classificação esperada:** **Pode prosseguir com restrição.**

Evidências:

- a base preparada contém **4.976 pares** observados de `espera_media_min` e `cupons_por_100_entradas`;
- existem **24 esperas ausentes** no total;
- na faixa 150–199 há **1.734 pares observados** e **7 esperas ausentes**;
- a cobertura de pares por loja na faixa varia de **114 a 296**.

**Restrição:** os resultados futuros se referirão aos registros em que a espera é conhecida. Não se pode presumir que os registros com espera ausente tenham o mesmo comportamento dos observados, nem preencher ausência com zero. Nesta aula não se calcula direção ou intensidade da associação.

## Q3 — Resultado comercial

### a) Margem bruta

**Classificação esperada:** **Pode prosseguir.** `margem_bruta_pct` está disponível para os **5.000 loja-dias**. Ela permite comparar a parcela da receita correspondente ao lucro bruto, mas não representa lucro líquido nem resultado final após despesas.

### b) Resultado após despesas operacionais

**Classificação esperada:** **Ainda depende de pendência.**

- `despesas_com_pessoal`: **36 ausências**;
- `outras_despesas_operacionais`: **42 ausências**;
- as duas colunas permanecem como `object`/texto;
- **14** valores de pessoal contêm `R$` e vírgula;
- **16** valores de outras despesas contêm vírgula decimal.

As contagens de formato não devem ser somadas às ausências. Ainda não há base consistente para calcular um resultado após essas despesas. Não introduzir conversão nesta aula.

### c) Registro R03047

`R03047` mantém `receita_liquida = 17.974,07`. O valor pode ser legítimo ou exigir verificação; ser muito diferente não demonstra erro. **Não aceitar exclusão automática.**

**Encaminhamento esperado:** ausências e possível valor atípico → **S8**; problemas de representação/formato das despesas, quando pertinentes ao objetivo analítico → **S9**.

## Q4 — Diagnóstico consolidado

Matriz esperada, aceitando redações equivalentes:

| Uso | Decisão esperada | Condição/limite principal | Próxima etapa |
|---|---|---|---|
| Loja + faixa 150–199 | Pode prosseguir | interpretar no nível do recorte | continuar exploração |
| Espera × aproveitamento | Pode prosseguir com restrição | registros com espera ausente limitam abrangência | S8 para ausências |
| Margem bruta | Pode prosseguir | não equivale a resultado final após despesas | continuar exploração |
| Resultado após despesas | Ainda depende de pendência | ausências e formatos mistos | S8 + S9 |

**Síntese esperada:** a base está adequada para alguns usos e condicionada ou insuficiente para outros. A validação deve ser por uso; “base limpa” ou “base pronta para tudo” é resposta inadequada.

Organização da equipe, perfil do público, mix de produtos, eventos locais e características do atendimento podem continuar relevantes para explicar diferenças, mas sua ausência como variáveis não deve ser registrada como valor ausente nem encaminhada automaticamente para tratamento.

## Q5 — Revisão entre duplas

A dupla revisora deve verificar a ligação entre **evidência → decisão → limite**, não apenas a redação ou a contagem. Exemplo de melhoria aceitável:

> “Podemos analisar espera” → “Podemos analisar os 1.734 pares observados na faixa, mas 7 esperas permanecem desconhecidas e limitam a abrangência.”

Aceite manutenção da resposta quando a dupla justificar por que sua evidência sustenta a classificação.

## Q6 — Resposta à equipe comercial

Exemplo de resposta:

> “A base permite aprofundar comparações entre lojas em uma faixa comum de movimento e investigar espera e aproveitamento usando os pares observados, com restrição pelos registros sem espera. A margem bruta pode ser comparada, mas o resultado após despesas ainda depende de ausências e formatos que precisam ser tratados. O valor de receita destacado também precisa ser verificado antes de qualquer decisão de tratamento.”

Aceite outras respostas que indiquem pelo menos:

1. um uso que pode prosseguir;
2. uma condição ou restrição;
3. uma pendência corretamente encaminhada.

Não aceitar como suficiente “a base está pronta”, “os dados estão limpos” ou “reduzir a espera vai melhorar o resultado”. A prioridade discutida na S6 não é critério de correção desta aula.

## Critérios de acompanhamento

Para evidenciar o nível **Analisar**, o estudante deve integrar **evidência, decisão e limite**. Executar células ou repetir números corretos sem relacioná-los ao uso pretendido indica necessidade de mediação.

Não solicite tratamento das pendências nem exportação de uma nova base para a próxima aula. Os materiais seguintes serão autossuficientes; a continuidade ocorre pela compreensão das decisões de adequação e pelos problemas corretamente encaminhados.

**Referências numéricas para conferência:** 5.000 registros após retirar a cópia; oito lojas com 625 registros cada; faixa 150–199 com 1.741 registros e mínimo 115 por loja; 1.734 pares espera × aproveitamento nessa faixa; 24 esperas ausentes no total e 7 na faixa.

Preserve os limites: associação não demonstra causalidade; intensidade de correlação não mede benefício de intervenção; margem bruta não é resultado final; o conjunto é simulado e não autoriza generalização para empresas reais.
